# Landsat M2M AOI (KML) — Jun/Aug 2016–2021

This notebook uses the USGS M2M API to search Landsat Collection 2 (L2) scenes for July & August of 2016–2021 inside a local AOI saved as KML. Edit the **CONFIG** cell to set your `APP_TOKEN` (recommended), `AOI_PATH`, and folders.

In [1]:
# Requirements (run once)
# !pip install requests shapely geopandas tqdm pandas fiona nbformat
# !pip install pystac_client shapely geopandas folium requests tqdm

In [2]:
# -------------------------
# IMPORTS
# -------------------------
import os
import requests
import csv
import geopandas as gpd
from shapely.geometry import mapping
from tqdm import tqdm
import folium
from calendar import monthrange
from collections import defaultdict
from pystac_client import Client

In [3]:
# -------------------------
# CONFIG
# -------------------------
NOTEBOOK_DIR = os.getcwd()
PROJECT_DIR = os.path.abspath(os.path.join(NOTEBOOK_DIR, ".."))

AOI_PATH = os.path.join(PROJECT_DIR, "data", "aoi", "2025_aoi.geojson")
RESULTS_DIR = os.path.join(PROJECT_DIR, "data", "raw", "landsat_stac")
LOG_CSV = os.path.join(RESULTS_DIR, "download_log.csv")
os.makedirs(RESULTS_DIR, exist_ok=True)

COLLECTION = "landsat-c2l2-sr"   # Collection 2, Level-2 Surface Reflectance
YEARS = [2016, 2017, 2018, 2019, 2020, 2021]
MONTHS = [6, 7, 8]  # June, July, August
MAX_CLOUD = 30

In [4]:
# -------------------------
# LOAD AOI
# -------------------------
aoi_gdf = gpd.read_file(AOI_PATH)
aoi_geom = aoi_gdf.unary_union  # combined geometry
print("AOI loaded:", aoi_gdf)

# -------------------------
# PLOT AOI for QA
# -------------------------
m = folium.Map(
    location=[aoi_gdf.geometry.centroid.y.mean(), aoi_gdf.geometry.centroid.x.mean()],
    zoom_start=6
)
folium.GeoJson(aoi_gdf).add_to(m)
m  # interactive map in notebook

AOI loaded:                         Name Description  \
0  57°15'33.1"N 98°10'45.8"W               
1                        aoi               

                                            geometry  
0               POINT Z (-98.17940 57.25920 0.00000)  
1  POLYGON Z ((-101.15335 56.13383 0.00000, -95.3...  


/Users/domenica/opt/anaconda3/lib/python3.8/site-packages/pyproj/crs/crs.py:141: FutureWarning: '+init=<authority>:<code>' syntax is deprecated. '<authority>:<code>' is the preferred initialization method. When making the change, be mindful of axis order changes: https://pyproj4.github.io/pyproj/stable/gotchas.html#axis-order-changes-in-proj-6
  in_crs_string = _prepare_from_proj_string(in_crs_string)


In [5]:
# -------------------------
# CONNECT TO USGS STAC
# -------------------------
stac_api = Client.open("https://landsatlook.usgs.gov/stac-server")

In [6]:
# -------------------------
# SEARCH & SELECT BEST SCENES PER YEAR
# -------------------------
# Will store the best scene per location per year
best_scenes_by_location = defaultdict(dict)

for year in YEARS:
    year_candidates = []
    
    # Gather all candidate scenes for the year
    for month in MONTHS:
        start_day = f"{year}-{month:02d}-01"
        _, last_day_num = monthrange(year, month)
        end_day = f"{year}-{month:02d}-{last_day_num}"
        
        search_result = stac_api.search(
            collections=[COLLECTION],
            intersects=mapping(aoi_geom),
            datetime=f"{start_day}/{end_day}",
            query={"eo:cloud_cover": {"lt": MAX_CLOUD}},
            limit=500
        )
        
        items = search_result.get_all_items()
        year_candidates.extend(items)
    
    # Group by scene location and select lowest-cloud scene
    for item in year_candidates:
        scene_key = item.id[:15]  # Adjust as needed for path/row grouping
        current_best = best_scenes_by_location[scene_key].get(year)
        cloud = item.properties.get("eo:cloud_cover", 100)
        if current_best is None or cloud < current_best.properties.get("eo:cloud_cover", 100):
            best_scenes_by_location[scene_key][year] = item

# Flatten dictionary for download
selected_scenes = []
for loc_scenes in best_scenes_by_location.values():
    selected_scenes.extend(loc_scenes.values())

print(f"\nTotal selected scenes: {len(selected_scenes)}")
for s in selected_scenes:
    print(s.id, s.properties.get("acquisitionDate"), s.properties.get("eo:cloud_cover"))

/Users/domenica/opt/anaconda3/lib/python3.8/site-packages/pystac_client/item_search.py:849: FutureWarning: get_all_items() is deprecated, use item_collection() instead.
  warnings.warn(



Total selected scenes: 154
LE07_L2SP_034019_20160816_20200902_02_T1_SR None 11
LE07_L2SP_034019_20170702_20200831_02_T1_SR None 0
LE07_L2SP_034019_20180822_20200828_02_T2_SR None 5
LE07_L2SP_034019_20190622_20200825_02_T1_SR None 28
LE07_L2SP_034019_20200827_20200924_02_T2_SR None 25
LE07_L2SP_034019_20210830_20210925_02_T1_SR None 3
LC08_L2SP_035021_20160831_20200906_02_T1_SR None 13.48
LC08_L2SP_035021_20170802_20200903_02_T1_SR None 0.2
LC08_L2SP_035021_20180805_20200831_02_T1_SR None 24.8
LC08_L2SP_035021_20190723_20200827_02_T1_SR None 5.41
LC08_L2SP_035020_20200623_20200823_02_T1_SR None 3.79
LC08_L2SP_035021_20210626_20210707_02_T1_SR None 0.98
LE07_L2SP_036021_20160627_20200902_02_T1_SR None 13
LE07_L2SP_036021_20170716_20200831_02_T1_SR None 10
LE07_L2SP_036020_20180703_20200829_02_T1_SR None 11
LE07_L2SP_036021_20190722_20201008_02_T1_SR None 0
LE07_L2SP_036020_20200622_20200822_02_T1_SR None 1
LE07_L2SP_036020_20210609_20210705_02_T1_SR None 29
LC08_L2SP_037020_20160626_202

In [7]:
# -------------------------
# DOWNLOAD FUNCTION WITH PROGRESS & LOG
# -------------------------
def download_scene(scene, out_dir=RESULTS_DIR, log_file=LOG_CSV):
    scene_id = scene.id
    acquisition_date = scene.properties.get("acquisitionDate", "UNKNOWN")
    cloud_cover = scene.properties.get("eo:cloud_cover", "")
    output_folder = os.path.join(out_dir, scene_id)

    if os.path.exists(output_folder):
        status = "skipped"
        print(f"Already downloaded {scene_id}")
    else:
        os.makedirs(output_folder, exist_ok=True)
        status = "downloaded"
        for asset_name, asset in scene.assets.items():
            href = asset.href
            out_file = os.path.join(output_folder, os.path.basename(href))
            print(f"\nDownloading {asset_name} -> {out_file}")

            # Stream download with progress bar
            with requests.get(href, stream=True) as r:
                r.raise_for_status()
                total = int(r.headers.get('Content-Length', 0))
                with open(out_file, 'wb') as f, tqdm(
                    desc=os.path.basename(out_file),
                    total=total,
                    unit='B',
                    unit_scale=True,
                    unit_divisor=1024
                ) as bar:
                    for chunk in r.iter_content(chunk_size=8192):
                        if chunk:
                            f.write(chunk)
                            bar.update(len(chunk))

    # Append log
    with open(log_file, "a", newline="") as f:
        writer = csv.writer(f)
        writer.writerow([scene_id, acquisition_date, cloud_cover, status])

# -------------------------
# CREATE LOG CSV HEADER
# -------------------------
if not os.path.exists(LOG_CSV):
    with open(LOG_CSV, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["scene_id", "acquisition_date", "cloud_cover", "status"])

# -------------------------
# DOWNLOAD ALL SELECTED SCENES
# -------------------------
for scene in selected_scenes:
    download_scene(scene)

KeyboardInterrupt: 